In [1]:
#to create a web server in Python

In [2]:
pip install fastapi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/") #?? why is there a slash inside here and /submit in post, what else we could do here??
async def get_ex():
    return{"message":"Hello world "}

@app.post("/submit") # /submit is the URL path that tells client which action to perform
async def post_ex():
    return {"message":"posted"}

# More actions are:
#@app.post("/predict") 
#@app.post("/v1/chat/completions")


# 2. Dynamic Path Parameters (variables inside the URL)
@app.get("/users/{user_id}")
async def get_user(user_id: int):
    return {user_id: user_id}

@app.get("/models/{model_name}/status")
async def model_status(model_name: str):
    return {"model": model_name, "status": "ready"}

@app.get("/models/{model_name}")
async def get_model_info(model_name: str):
    return {
        "model_requested": model_name,
        "is_supported": model_name in ["gpt-4o", "claude-3-5-sonnet", "llama-3"]
    }

In [11]:
client = TestClient(app)
print("Server will return **HTTP Response object**. when using cliend.get(), client.post()")
res_get = client.get("/")
print("GET Result:", res_get.status_code, res_get.json())

res_mismatch = client.post("/")
print("Mismatch result:", res_mismatch.status_code, res_mismatch.json())

# 'json = ' is used to convert py dict to json string and alerts the server that it is sending JSON data.
res_post = client.post("/submit", json ={"user" : "dev_user"}) 
print("Post result", res_post.status_code, res_post.json())

print(client.get("/models/random-model").json())
print(client.get("/models/gpt-4o").json())


Server will return **HTTP Response object**. when using cliend.get(), client.post()
GET Result: 200 {'message': 'Hello world '}
Mismatch result: 405 {'detail': 'Method Not Allowed'}
Post result 200 {'message': 'posted'}
{'model_requested': 'random-model', 'is_supported': False}
{'model_requested': 'gpt-4o', 'is_supported': True}


In [12]:
# It looks like the line breaks got stripped during the paste, turning multiple statements into one long line!

# Here i/s the clean snippet formatted with explicit newlines. You can copy this directly into your Jupyter cell:
# 
# ```python
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI()

# 1. Define schema
class UserProfile(BaseModel):
    username: str
    tier: str = "free"
    token_limit: int = Field(gt=0)

# 2. Endpoint with validation
@app.post("/register")
async def register_user(profile: UserProfile):
    return {
        "status": "success",
        "saved_user": profile.username,
        "tokens": profile.token_limit
    }

# 3. Test client
client = TestClient(app)

# Test 1: Valid input (200 OK)
res_valid = client.post(
    "/register", 
    json={"username": "alex", "token_limit": 5000}
)
print("Test 1 (Valid):", res_valid.status_code, res_valid.json())

# Test 2: Invalid input (token_limit <= 0 triggers 422)
res_invalid = client.post(
    "/register", 
    json={"username": "alex", "token_limit": -10}
)
print("Test 2 (Invalid):", res_invalid.status_code, res_invalid.json())


# Give this a run—you should see Test 1 succeed with `200` and Test 2 fail cleanly with `422 Unprocessable Entity`!

Test 1 (Valid): 200 {'status': 'success', 'saved_user': 'alex', 'tokens': 5000}
Test 2 (Invalid): 422 {'detail': [{'type': 'greater_than', 'loc': ['body', 'token_limit'], 'msg': 'Input should be greater than 0', 'input': -10, 'ctx': {'gt': 0}}]}
